<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Speech Commands Keyword Spotting (KWS) — Akida Pico

Run time: about 15 minutes with training skipped (the default); full training adds a few hours on a GPU.

This notebook takes a keyword spotting model from raw audio to a model that maps onto **Akida Pico**:

1. Look at the data: raw 16 kHz audio clips, 12 classes
2. Build the recurrent TENN, in its training form
3. Train it (or load the trained model)
4. Convert it to its **stateful** form, which processes audio chunk by chunk
5. Quantize it to 8 bits with `quantizeml`
6. Convert it to Akida with `cnn2snn`, evaluate it, and map it onto the Akida Pico virtual device

The data loading, model and training code live in the example's scripts
(`speech_commands_data.py`, `speech_commands_model.py`, `speech_commands_train.py`,
`speech_commands_eval.py`); this notebook shows the Akida Pico specific steps.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '1')
os.environ['ASSERT_ENABLED'] = '0'   # quantizeml's internal assertions aren't needed here

DATA_PATH = './data/speech_commands'   # TFDS data_dir; downloaded on first use
MODELS_DIR = './pretrained_models'
SAMPLES_PATH = './data/sc12_batch100_1024samples.npz'

# Full training takes a few hours on a GPU. By default, load the trained model instead.
TRAIN = False

# Streaming evaluation is slower than whole-clip evaluation, so the notebook evaluates the
# stateful, quantized and Akida models on the first NUM_EVAL test clips. The README
# reports the full test split, from speech_commands_eval.py.
NUM_EVAL = 978

## 1. The data

The model reads **raw audio**: each 1-second clip at 16 kHz is padded or cropped to
16,384 samples and given to the model as int16 values. There is no MFCC or other
feature extraction in front of the model, unlike the Akida 1 and Akida 2 Speech
Commands examples.

In [ ]:
from speech_commands_data import CLASS_NAMES, get_test_data

x_test, y_test = get_test_data(DATA_PATH)
print(f'Test split: {x_test.shape[0]} clips of {x_test.shape[1]} samples')
print('Clips per class:', dict(zip(CLASS_NAMES, np.bincount(y_test).tolist())))

fig, axes = plt.subplots(1, 3, figsize=(12, 2.5), sharey=True)
for ax, cls in zip(axes, ['yes', 'stop', '_silence_']):
    i = int(np.where(y_test == CLASS_NAMES.index(cls))[0][0])
    ax.plot(np.arange(x_test.shape[1]) / 16000, x_test[i, :, 0], lw=0.5)
    ax.set_title(cls)
    ax.set_xlabel('time (s)')
axes[0].set_ylabel('int16 sample')
plt.tight_layout()
plt.show()

## 2. The model, in its training form

`tenn_recurrent_sc12` from `akida_models` stacks six recurrent TENN blocks (8 to 128
channels), each subsampling time by 4 or 2, and a 12-class recurrent head. A
`Rescaling` layer (divide by 2<sup>15</sup>) is part of the model, so it takes the raw
int16 values.

In this form each recurrent layer is a `Kernelized` layer: a temporal convolution
whose kernel is generated from a small set of state-space parameters. Over a whole clip
this computes efficiently on a GPU, which is why the model is trained this way.

In [ ]:
from speech_commands_model import build_speech_commands_model

model = build_speech_commands_model()
model.summary()

## 3. Training

The recipe is the one `akida_models` uses for this model: 200 epochs of AdamW, with a
learning rate warm-up then cosine decay, and class-balanced epochs (the training split
is dominated by `_unknown_`). The best epoch is picked on the validation split by
macro-F1. Set `TRAIN = True` above to run it here, or run the script:

```bash
python speech_commands_train.py -l models/tenn_recurrent_sc12_untrained.h5 -s models/tenn_recurrent_sc12.h5 -e 200
```

In [ ]:
from tf_keras.models import load_model

if TRAIN:
    from speech_commands_data import get_data
    from speech_commands_train import train_speech_commands
    train_ds, val_ds, train_steps, val_steps = get_data(DATA_PATH, batch_size=128,
                                                        timestep=model.input_shape[1])
    train_speech_commands(model, train_ds, val_ds, train_steps, val_steps, epochs=200)
else:
    model = load_model(os.path.join(MODELS_DIR, 'tenn_recurrent_sc12.h5'))

In [ ]:
from speech_commands_eval import evaluate

acc, f1 = evaluate(model, x_test, y_test)
print(f'Float model, whole clips, full test split: accuracy {acc * 100:.2f}%, macro-F1 {f1:.4f}')

## 4. Stateful form: streaming, chunk by chunk

Akida Pico doesn't see a whole clip at once. `convert_to_stateful` turns every
`Kernelized` layer into a `StatefulRecurrent` layer **with the same weights**. The
stateful model reads 256 samples (16 ms) at a time and keeps each layer's internal
state between calls, so it can run on a continuous audio stream.

Its input shape is now one chunk, and it produces one prediction per chunk. To classify
a clip, the evaluation feeds its 64 chunks in order and averages the per-chunk logits.

In [ ]:
from speech_commands_model import to_stateful

stateful = to_stateful(model, timesteps=256)
stateful.summary()

x_eval, y_eval = x_test[:NUM_EVAL], y_test[:NUM_EVAL]
acc, f1 = evaluate(stateful, x_eval, y_eval)
print(f'Float stateful model, {NUM_EVAL} clips: accuracy {acc * 100:.2f}%, macro-F1 {f1:.4f}')

## 5. Quantization

The stateful model is quantized with `quantizeml`: 8-bit weights and activations, no
quantization-aware training. The inputs are **int16** (`input_dtype='int16'`): Akida
Pico's first layer accepts 16-bit inputs, so the raw audio samples go straight in.
Calibration uses real audio: BrainChip's published set of int16 chunks from the
training split.

In [ ]:
from tf_keras.utils import get_file
from quantizeml.layers import QuantizationParams
from quantizeml.models import quantize
from speech_commands_data import SAMPLES_URL, get_samples

if not os.path.exists(SAMPLES_PATH):
    get_file(os.path.abspath(SAMPLES_PATH), SAMPLES_URL)
samples = get_samples(SAMPLES_PATH)
print('Calibration samples:', samples.shape, samples.dtype)

qparams = QuantizationParams(input_dtype='int16', weight_bits=8, activation_bits=8)
quantized = quantize(stateful, qparams=qparams, samples=samples, epochs=1, batch_size=100)

acc, f1 = evaluate(quantized, x_eval, y_eval)
print(f'8-bit quantized model, {NUM_EVAL} clips: accuracy {acc * 100:.2f}%, macro-F1 {f1:.4f}')

## 6. Conversion to Akida, and mapping onto Akida Pico

`cnn2snn.convert` turns the quantized model into an Akida model. Without a device
attached, the Akida runtime evaluates it in software.

In [ ]:
from cnn2snn import convert

akida_model = convert(quantized)
akida_model.summary()

acc, f1 = evaluate(akida_model, x_eval, y_eval)
print(f'Akida model, {NUM_EVAL} clips: accuracy {acc * 100:.2f}%, macro-F1 {f1:.4f}')

Mapping places the model's layers on the hardware. `akida.PicoIP()` is a **virtual**
Akida Pico device: mapping onto it checks the model against the
[Akida Pico hardware limits](https://doc.brainchipinc.com/user_guide/hardware/pico.html)
(at most 8 recurrent layers, 16-bit inputs up to 128 channels, subsampling at most 4
per layer, unbounded ReLU, and so on) without a board. A model mapped onto a virtual
device can't run inference, so `pico_mapping` maps a copy.

In [ ]:
from speech_commands_eval import pico_mapping

print(pico_mapping(akida_model))

The whole model runs as hardware on Akida Pico, with no part left to software.

## 7. Hardware benchmark

📝 Planned: latency and power on the Akida Pico FPGA are tracked in
[#104](https://github.com/Brainchip-Inc/brainchip_devhub/issues/104).